# Task 1.2: Key Assumptions

**Paper**: *A Simple Geometric Interpretation of SVM using Stochastic Adversaries* — Livni, Crammer, Globerson (AISTATS 2012)

---

Below I identify three assumptions that are specifically embedded in the RSVM2 framework proposed in this paper. These are not generic ML assumptions — each one is structurally required by the derivation or the formulation itself.

## Assumption 1: Zero-Mean Perturbation Constraint ($\mathbb{E}[\tilde{x}] = x$)

**Assumption Statement:**
The stochastic adversary must pick perturbation distributions whose expected value equals the original data point $x$. Formally, $\mathbb{E}_{p(\tilde{x}|x)}[\tilde{x}] = x$. In other words, the noise the adversary injects is always "centered" around the true point — it can spread things out, but it cannot systematically shift them.

**Why the Method Needs It:**
When I traced through the proof of Theorem 3.1 (Equations 6–7), I noticed that the variable substitution $\tilde{x} \leftarrow \tilde{x} - x$ only works cleanly because the mean constraint pins $\mathbb{E}[\tilde{x}]$ to $x$. This is what makes the perturbation a zero-mean random variable after the change of variables, which then lets the dual reformulation go through and produce the neat $\ell_{2,\infty}$ regularizer. Without this centering constraint, the adversary could just translate every point to some single worst-case location, and the minimax problem would degenerate — the adversary would be too powerful for the framework to yield anything useful.

**Violation Scenario:**
Consider real-world sensor data where instruments have a systematic calibration drift — say a temperature sensor that consistently reads 2°C too high. The perturbation noise here is *not* zero-mean; it has a persistent directional bias. In this setting, RSVM2 would underestimate the adversary's actual power because it assumes the average perturbed point coincides with the original, which simply isn't true when there's systematic bias in the corruption.

**Paper Reference:**
Section 2, definition of $S(x; \sigma, f)$ — the constraint $\mathbb{E}_{p(\tilde{x}|x)}[\tilde{x}] = x$ appears explicitly as the first condition defining the adversary's allowed perturbation set.

## Assumption 2: Uniform Perturbation Budget Across All Training Points

**Assumption Statement:**
Every training point $x_i$ is subject to the *same* perturbation budget $\sigma$. The adversary set $S(x_i; \sigma)$ uses one shared scalar $\sigma$ for all $i$ — no point gets a larger or smaller noise budget than any other.

**Why the Method Needs It:**
The equivalence result in Theorem 3.1 relies on summing individual per-point adversary sub-problems that all share the same $\sigma$. I worked through what happens if you try to give each point its own $\sigma_i$: the regularization term no longer factors out as a single global $\sigma \cdot \max_y \|w_y\|_2$. Instead you'd get a messy point-dependent penalty that couples the optimization over $w$ with the per-sample noise levels. The clean connection to standard SVM regularization breaks down entirely. It's also worth noting that in Section 5, the heuristic for choosing $\sigma$ computes one global value from the average nearest-neighbor distance — reinforcing that the framework is built around a single shared budget.

**Violation Scenario:**
Think about a medical dataset where measurements from young, healthy patients are very stable (low noise), but measurements from elderly patients are highly variable (high noise). This is textbook heteroscedastic noise — different regions of the feature space have fundamentally different noise levels. Forcing a single $\sigma$ would over-regularize in the low-noise region (young patients) and under-regularize in the high-noise region (elderly patients), leading to a classifier that's suboptimal in both regimes.

**Paper Reference:**
Section 2, Equation 3 — $\sigma$ appears as a single scalar shared across all $i$ in the outer summation. Also Section 5, where a single global $\sigma$ is computed via the average nearest-neighbor heuristic.

## Assumption 3: Linear Decision Boundaries

**Assumption Statement:**
The classifier family is restricted to linear models of the form $\hat{y} = \arg\max_y \, w_y \cdot x$, where each class $y$ has a weight vector $w_y \in \mathbb{R}^d$. Decision boundaries between any pair of classes are hyperplanes, and the model interacts with inputs purely through inner products.

**Why the Method Needs It:**
This is the assumption I found most structurally critical. The entire dual derivation in Theorem 3.1 hinges on the fact that the multiclass hinge loss is *piecewise linear* in $\tilde{x}$ (see Equations 7–9). Because $w_{\bar{y}} \cdot \tilde{x}$ is linear in $\tilde{x}$, the inner maximization over the adversary's distribution can be reformulated as a clean convex program with linear constraints plus a single norm term from the variance budget. If you swapped in a nonlinear classifier — say a neural network — the loss landscape in $\tilde{x}$ would become non-convex, and the dual that produces the $\ell_{2,\infty}$ norm regularizer simply wouldn't go through. The geometric interpretation (Section 4) also depends on linearity, since the directional derivative argument assumes linear separators.

**Violation Scenario:**
A dataset with concentric ring-shaped classes (like `make_circles` in scikit-learn) cannot be separated by any linear classifier regardless of regularization. RSVM2 would fail outright on such data without first mapping it into a higher-dimensional feature space via a kernel. Notably, the paper does not discuss kernel extensions of RSVM2 — the framework as presented is purely linear.

**Paper Reference:**
Section 2: "we consider linear models parameterized by a set of $L$ weight vectors $w_y \in \mathbb{R}^d$". The proof in Section 3 (Equations 7–12) fundamentally uses the linearity of $w_{\bar{y}} \cdot \tilde{x}$ in $\tilde{x}$ at every step of the dual derivation.